In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -q diffusers transformers accelerate gradio rembg pillow numpy onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.7 MB/s eta 0:00:00


In [3]:
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16
).to("cuda")

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [4]:
def enhance_prompt(prompt, style):
    base = "ultra realistic, 4k, high detail, sharp focus, professional product photography"
    
    styles = {
        "Amazon": "white background, centered product, soft shadow, e-commerce",
        "Luxury": "dark background, dramatic lighting, premium, cinematic",
        "Minimal": "clean minimal background, soft light, modern aesthetic",
        "Studio": "studio lighting setup, DSLR quality, high sharpness"
    }
    
    return f"{prompt}, {base}, {styles.get(style, '')}"

In [5]:
import random

def generate_images(prompt, style, steps, seed):
    final_prompt = enhance_prompt(prompt, style)
    
    generator = torch.manual_seed(seed)
    
    images = pipe(
        final_prompt,
        negative_prompt="blurry, distorted, low quality, watermark",
        num_inference_steps=steps,
        num_images_per_prompt=3,
        generator=generator
    ).images
    
    return images

In [6]:
import numpy as np

def select_best(images):
    # Simple sharpness proxy (variance)
    scores = []
    
    for img in images:
        arr = np.array(img)
        score = arr.var()
        scores.append(score)
    
    best_index = np.argmax(scores)
    return images[best_index]

In [7]:
from rembg import remove

def remove_bg(image):
    return remove(image)

In [8]:
def pipeline(prompt, style, steps, seed, remove_background):
    images = generate_images(prompt, style, steps, seed)
    
    best_image = select_best(images)
    
    if remove_background:
        best_image = remove_bg(best_image)
    
    best_image.save("/kaggle/working/final_output.png")
    
    return images, best_image

In [9]:
import gradio as gr

demo = gr.Interface(
    fn=pipeline,
    inputs=[
        gr.Textbox(label="Product Description"),
        gr.Dropdown(["Amazon", "Luxury", "Minimal", "Studio"], label="Style"),
        gr.Slider(20, 50, value=30, label="Quality Steps"),
        gr.Number(value=42, label="Seed (Reproducibility)"),
        gr.Checkbox(label="Remove Background")
    ],
    outputs=[
        gr.Gallery(label="All Generated Images"),
        gr.Image(label="Best Selected Image")
    ],
    title="🚀 AI Product Image Generator PRO MAX",
    description="Advanced AI system with prompt engineering, multi-generation, ranking, and background removal"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://7164da974c14cc31c9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## AI Product Image Generator 

This project simulates a real-world AI-powered product photography pipeline.

### Advanced Features:
- Stable Diffusion XL for high-quality generation
- Intelligent prompt enhancement system
- Multi-image generation with seed control (reproducibility)
- Automated best image selection using statistical sharpness
- Background removal using AI
- Interactive UI with advanced controls

### Innovation:
Unlike basic generators, this system includes a ranking mechanism to automatically select the best product image, mimicking real-world AI pipelines used in e-commerce platforms.

### Applications:
- E-commerce product listing
- Automated marketing content
- Digital product visualization

### Conclusion:
This project demonstrates how generative AI can automate and enhance product photography workflows with minimal human effort.